# feeder_court - training

Trains `yolov8n-p2` (stride-4 head, per the Phase 0 gate decision: `p2`) on
hand-corrected OV9281 court footage.

- **Split**: train = `near` + `mid` (41 frames), val = `far` (12 frames, held out).
- **Never downscale.** Median shuttle is 13.8 px on the worst clip; at imgsz 640
  that becomes ~7 px and at 320 it is gone. `imgsz=1280` is the native long side.
- **Mono footage**: hue/saturation augmentation disabled - those channels carry
  no information on this camera.
- `scale=0.25` instead of the 0.5 default, which would annihilate a 13 px object.

Runtime -> Change runtime type -> **T4 GPU** before running.


In [ ]:
!nvidia-smi
!pip -q install ultralytics==8.4.14

## Upload the dataset

Run this cell, then choose `feeder_court_yolo.zip` (the local
`setup/datasets/feeder_court_yolo` tree, zipped).

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
import glob, os, zipfile, yaml

ZIP = "feeder_court_yolo.zip"
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall("/content")

ROOT = "/content/feeder_court_yolo"
DATA = f"{ROOT}/data.yaml"
with open(DATA, "w") as fh:
    yaml.safe_dump({"path": ROOT, "train": "train/images", "val": "val/images",
                    "nc": 1, "names": ["shuttlecock"]}, fh, sort_keys=False)
print(open(DATA).read())

In [ ]:
# Refuse to spend GPU time on a leaked split.
train_imgs = sorted(glob.glob(f"{ROOT}/train/images/*.jpg"))
val_imgs   = sorted(glob.glob(f"{ROOT}/val/images/*.jpg"))

leaked = [p for p in train_imgs if os.path.basename(p).startswith("far_")]
assert not leaked, f"far frames leaked into train: {leaked[:5]}"
assert all(os.path.basename(p).startswith("far_") for p in val_imgs), "val is not pure far"
assert len(train_imgs) == 41 and len(val_imgs) == 12, (len(train_imgs), len(val_imgs))

from collections import Counter
print("train:", Counter(os.path.basename(p).split("_")[0] for p in train_imgs))
print("val:  ", Counter(os.path.basename(p).split("_")[0] for p in val_imgs))
print("OK: no far frames in train")

In [ ]:
from ultralytics import YOLO

IMGSZ = 1280  # native long side - do NOT lower this

model = YOLO("yolov8-p2.yaml")

results = model.train(
    data="/content/feeder_court_yolo/data.yaml",
    epochs=80,
    imgsz=IMGSZ,
    batch=8,
    workers=2,
    seed=0,
    deterministic=True,
    project="runs/feeder_court",
    name="p2-native",
    patience=20,
    cache=False,
    # --- mono-specific augmentation ---
    hsv_h=0.0,     # no hue information exists in this footage
    hsv_s=0.0,     # no saturation information exists either
    scale=0.25,    # default 0.5 would annihilate a 13 px object
    flipud=0.0,    # gravity is real; shuttles fall
    fliplr=0.5,    # the court is roughly symmetric
    mosaic=1.0,
    close_mosaic=10,
    plots=True,
    val=True,
)

## Download

`best.pt` is selected on the `far` validation mAP, which is also the held-out
evaluation clip - so its score there is optimistic. `last.pt` ships alongside it
for an uncontaminated checkpoint, and `results.csv` carries the per-epoch curve.

In [ ]:
import glob, os, shutil
from google.colab import files

# Ultralytics increments the run name (p2-native2, p2-native3, ...) if the
# training cell ran more than once, so never hardcode the path.
RUN = ""
try:
    RUN = str(model.trainer.save_dir)
except Exception:
    pass

if not RUN or not os.path.isdir(RUN):
    hits = glob.glob("/content/**/weights/best.pt", recursive=True)
    assert hits, "no best.pt anywhere under /content - did the training cell finish?"
    RUN = os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))

print("run dir:", RUN)
print(sorted(os.listdir(RUN)))
print(sorted(os.listdir(os.path.join(RUN, "weights"))))

shutil.make_archive("/content/feeder_court_train", "zip", RUN)
files.download("/content/feeder_court_train.zip")